# Preprocessing Dataset NHANES 2015-2016 untuk Prediksi Stroke

Notebook ini digunakan untuk mengunduh dataset NHANES 2015-2016, melakukan ekstraksi, penggabungan, dan preprocessing data untuk pemodelan prediksi stroke. 

Fitur gaya hidup (tidur, stres, dan aktivitas fisik) disamakan masing-masing berjumlah 5 fitur untuk menjaga keseimbangan kontribusi variabel gaya hidup.

## Alur Preprocessing
1. Install dan Import Library
2. Download dan Load Dataset NHANES
3. Penggabungan Dataset berdasarkan SEQN
4. Preprocessing Data:
   - Pembersihan Data (Data Cleaning)
   - Penanganan Nilai Hilang (Missing Value Handling)
   - Penyandian Fitur (Feature Encoding)
   - Penskalaan Fitur (Feature Scaling)
   - Resampling menggunakan SMOTE

## 1. Install dan Import Library

In [1]:
# Install library tambahan
!pip install imbalanced-learn xgboost shap -q

import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE

print("Import library berhasil.")


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Import library berhasil.


## 2. Download dan Load Dataset NHANES 2015-2016

In [2]:
# Mengunduh dataset dari repository Github jika belum ada
if not os.path.exists('NHANES-Stroke-Dataset'):
    os.system('git clone https://github.com/Adityamulyaf/NHANES-Stroke-Dataset.git')

# Menentukan path folder dataset
if os.path.exists('content/'):
    DATA_PATH = 'content/'
elif os.path.exists('NHANES-Stroke-Dataset/content/'):
    DATA_PATH = 'NHANES-Stroke-Dataset/content/'
else:
    DATA_PATH = '/content/NHANES-Stroke-Dataset/content/'

print("Path dataset yang digunakan:", DATA_PATH)
print("Daftar file dataset:")
for file in os.listdir(DATA_PATH):
    print(f" - {file}")

Path dataset yang digunakan: content/
Daftar file dataset:
 - BMX_I.xpt
 - BPQ_I.xpt
 - BPX_I.xpt
 - DEMO_I.xpt
 - DIQ_I.xpt
 - DPQ_I.xpt
 - MCQ_I.xpt
 - PAQ_I.xpt
 - SLQ_I.xpt
 - SMQ_I.xpt


In [3]:
# Memuat file SAS XPORT (.xpt)
dfs = {}
files = {
    'demo': 'DEMO_I.xpt',
    'mcq':  'MCQ_I.xpt',
    'bpq':  'BPQ_I.xpt',
    'diq':  'DIQ_I.xpt',
    'bmx':  'BMX_I.xpt',
    'bpx':  'BPX_I.xpt',
    'smq':  'SMQ_I.xpt',
    'slq':  'SLQ_I.xpt',
    'dpq':  'DPQ_I.xpt',
    'paq':  'PAQ_I.xpt',
}

for name, filename in files.items():
    path = os.path.join(DATA_PATH, filename)
    print(f"Loading {filename}...")
    try:
        dfs[name] = pd.read_sas(path, format='xport', encoding='utf-8')
        print(f"Berhasil memuat {filename}: {dfs[name].shape[0]} baris, {dfs[name].shape[1]} kolom")
    except Exception as e:
        print(f"Gagal memuat {filename}: {e}")

Loading DEMO_I.xpt...
Berhasil memuat DEMO_I.xpt: 9971 baris, 47 kolom
Loading MCQ_I.xpt...
Berhasil memuat MCQ_I.xpt: 9575 baris, 90 kolom
Loading BPQ_I.xpt...
Berhasil memuat BPQ_I.xpt: 6327 baris, 11 kolom
Loading DIQ_I.xpt...
Berhasil memuat DIQ_I.xpt: 9575 baris, 54 kolom
Loading BMX_I.xpt...
Berhasil memuat BMX_I.xpt: 9544 baris, 26 kolom
Loading BPX_I.xpt...
Berhasil memuat BPX_I.xpt: 9544 baris, 21 kolom
Loading SMQ_I.xpt...
Berhasil memuat SMQ_I.xpt: 7001 baris, 42 kolom
Loading SLQ_I.xpt...
Berhasil memuat SLQ_I.xpt: 6327 baris, 8 kolom
Loading DPQ_I.xpt...
Berhasil memuat DPQ_I.xpt: 5735 baris, 11 kolom
Loading PAQ_I.xpt...
Berhasil memuat PAQ_I.xpt: 9255 baris, 94 kolom


## 3. Seleksi Kolom dan Penggabungan Dataset

In [4]:
# Seleksi kolom yang relevan dari masing-masing dataset

# Demografi
demo_cols = ['SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'DMDEDUC2', 'INDFMPIR']
demo_clean = dfs['demo'][demo_cols].copy()

# Medis dan Target (Stroke)
mcq_cols = ['SEQN', 'MCQ160F', 'MCQ160B', 'MCQ160C', 'MCQ160E']
mcq_clean = dfs['mcq'][mcq_cols].copy()

# Hipertensi dan Diabetes
bpq_cols = ['SEQN', 'BPQ020']
bpq_clean = dfs['bpq'][bpq_cols].copy()

diq_cols = ['SEQN', 'DIQ010']
diq_clean = dfs['diq'][diq_cols].copy()

# Pemeriksaan Fisik
bmx_cols = ['SEQN', 'BMXBMI', 'BMXWAIST']
bmx_clean = dfs['bmx'][bmx_cols].copy()

bpx_cols = ['SEQN', 'BPXSY1', 'BPXDI1']
bpx_clean = dfs['bpx'][bpx_cols].copy()

# Kebiasaan Merokok
smq_cols = ['SEQN', 'SMQ020', 'SMQ040']
smq_clean = dfs['smq'][smq_cols].copy()

# Fitur Tidur (5 Fitur)
slq_cols = ['SEQN', 'SLD012', 'SLQ030', 'SLQ040', 'SLQ050', 'SLQ120']
slq_clean = dfs['slq'][slq_cols].copy()

# Fitur Stres (5 Fitur)
dpq_cols = ['SEQN'] + [f'DPQ0{i}0' for i in range(1, 10)]
dpq_clean = dfs['dpq'][dpq_cols].copy()

# Pembersihan nilai kuesioner stres (hanya nilai 0-3 yang valid, selebihnya NaN)
phq_items = [f'DPQ0{i}0' for i in range(1, 10)]
for col in phq_items:
    dpq_clean[col] = dpq_clean[col].apply(lambda x: x if x <= 3.0 else np.nan)

# Membuat stress_score (skor total depresi/stres)
dpq_clean['stress_score'] = dpq_clean[phq_items].apply(
    lambda row: row.sum() if row.notna().sum() >= 7 else np.nan, axis=1
)

# Ganti nama 4 fitur kuesioner terpilih agar lebih deskriptif
dpq_rename = {
    'DPQ010': 'stress_anhedonia',
    'DPQ020': 'stress_depressed',
    'DPQ040': 'stress_fatigue',
    'DPQ060': 'stress_self_esteem'
}
dpq_clean = dpq_clean.rename(columns=dpq_rename)
stress_cols = ['SEQN', 'stress_score', 'stress_anhedonia', 'stress_depressed', 'stress_fatigue', 'stress_self_esteem']
dpq_clean = dpq_clean[stress_cols]

# Fitur Aktivitas Fisik (5 Fitur)
paq_cols = ['SEQN', 'PAQ605', 'PAD615', 'PAQ650', 'PAQ665', 'PAD680']
paq_clean = dfs['paq'][paq_cols].copy()

# Menggabungkan semua dataset berdasarkan nomor responden (SEQN)
df = demo_clean
for other in [mcq_clean, bpq_clean, diq_clean, bmx_clean,
              bpx_clean, smq_clean, slq_clean, dpq_clean, paq_clean]:
    df = df.merge(other, on='SEQN', how='left')

print(f"Total baris setelah penggabungan: {df.shape[0]}")
print(f"Total kolom setelah penggabungan: {df.shape[1]}")

Total baris setelah penggabungan: 9971
Total kolom setelah penggabungan: 33


## 4. Preprocessing Data

### A. Data Cleaning dan Feature Encoding

In [5]:
# Pembuatan variabel target (Stroke)
# MCQ160F: 1 = pernah stroke, 2 = tidak pernah. Konversi ke biner 1 dan 0.
df['stroke'] = df['MCQ160F'].map({1.0: 1, 2.0: 0})

# Hapus baris yang tidak memiliki label target (stroke)
df = df[df['stroke'].notna()].copy()
print("Distribusi awal variabel stroke:")
print(df['stroke'].value_counts())
print(f"Rasio stroke positif: {df['stroke'].mean():.2%}")

# Filter hanya responden berusia dewasa (18 tahun ke atas)
df = df[df['RIDAGEYR'] >= 18].copy()
print(f"Jumlah data setelah filter usia dewasa: {len(df)} baris")

Distribusi awal variabel stroke:
stroke
0.0    5505
1.0     209
Name: count, dtype: int64
Rasio stroke positif: 3.66%
Jumlah data setelah filter usia dewasa: 5714 baris


In [6]:
# Encoding variabel kategorik dan biner ke format 1/0
binary_cols = {
    'BPQ020': 'hypertension',    # 1=Ya, 2=Tidak
    'DIQ010': 'diabetes',         # 1=Ya, 2=Tidak (nilai 3=Borderline dikelompokkan ke 1)
    'MCQ160B': 'heart_failure',
    'MCQ160C': 'coronary_disease',
    'MCQ160E': 'heart_attack',
    'SMQ020': 'ever_smoked',
    'SLQ050': 'sleep_problem_doctor',
    'PAQ605': 'vigorous_work',
    'PAQ650': 'vigorous_leisure',
    'PAQ665': 'moderate_leisure',
}

for old, new in binary_cols.items():
    if old in df.columns:
        df[new] = df[old].map({1.0: 1, 2.0: 0, 3.0: 1})

# Mengubah nama kolom agar lebih deskriptif
rename_map = {
    'RIDAGEYR': 'age',
    'RIAGENDR': 'gender',          # 1=Laki-laki, 2=Perempuan
    'RIDRETH3': 'race',
    'DMDEDUC2': 'education',
    'INDFMPIR': 'income_ratio',
    'BMXBMI':   'bmi',
    'BMXWAIST': 'waist_circ',
    'BPXSY1':   'systolic_bp',
    'BPXDI1':   'diastolic_bp',
    'SMQ040':   'current_smoker',  # 1=setiap hari, 2=kadang-kadang, 3=tidak sama sekali
    'SLD012':   'sleep_hours',
    'SLQ030':   'snoring_freq',
    'SLQ040':   'sleep_apnea',
    'SLQ120':   'daytime_sleepy',
    'PAD615':   'vigorous_work_min',
    'PAD680':   'sedentary_min',
}

df = df.rename(columns=rename_map)

# Konversi gender: 1=Laki-laki -> 1, 2=Perempuan -> 0
df['gender'] = df['gender'].map({1.0: 1, 2.0: 0})

print("Encoding dan pemetaan fitur selesai.")

Encoding dan pemetaan fitur selesai.


### B. Pengelompokan Fitur

Mendefinisikan variabel-variabel ke dalam kelompok masing-masing. Di sini, jumlah fitur gaya hidup (tidur, stres, dan aktivitas fisik) telah disamakan masing-masing berjumlah 5 fitur.

In [7]:
# Fitur Klinis Dasar (Baseline)
CLINICAL = [
    'age', 'gender', 'race', 'education', 'income_ratio',
    'bmi', 'waist_circ', 'systolic_bp', 'diastolic_bp',
    'hypertension', 'diabetes', 'heart_failure',
    'coronary_disease', 'heart_attack',
    'ever_smoked', 'current_smoker'
]

# Fitur Tidur (5 Fitur)
SLEEP = [
    'sleep_hours', 'snoring_freq', 'sleep_apnea',
    'sleep_problem_doctor', 'daytime_sleepy'
]

# Fitur Stres (5 Fitur)
STRESS = [
    'stress_score', 'stress_anhedonia', 'stress_depressed',
    'stress_fatigue', 'stress_self_esteem'
]

# Fitur Aktivitas Fisik (5 Fitur)
PHYSICAL = [
    'vigorous_work', 'vigorous_work_min', 'vigorous_leisure',
    'moderate_leisure', 'sedentary_min'
]

TARGET = 'stroke'

# Memilih kolom yang digunakan dalam pemodelan
ALL_COLS = CLINICAL + SLEEP + STRESS + PHYSICAL + [TARGET]
df_model = df[ALL_COLS].copy()

print(f"Total fitur yang digunakan: {len(ALL_COLS) - 1}")
print(f" - Fitur Klinis: {len(CLINICAL)}")
print(f" - Fitur Tidur: {len(SLEEP)}")
print(f" - Fitur Stres: {len(STRESS)}")
print(f" - Fitur Aktivitas Fisik: {len(PHYSICAL)}")

Total fitur yang digunakan: 31
 - Fitur Klinis: 16
 - Fitur Tidur: 5
 - Fitur Stres: 5
 - Fitur Aktivitas Fisik: 5


### C. Penanganan Nilai Hilang (Missing Value Handling)

In [8]:
print("Jumlah missing values sebelum penanganan:")
missing_counts = df_model.isnull().sum()
print(missing_counts[missing_counts > 0])

# 1. Hapus baris yang missing di fitur klinis utama
core_cols = ['age', 'bmi', 'systolic_bp', 'hypertension', 'diabetes', TARGET]
df_model = df_model.dropna(subset=core_cols)

# 2. Imputasi nilai median untuk kolom numerik sisanya
num_cols = df_model.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != TARGET]

imputer = SimpleImputer(strategy='median')
df_model[num_cols] = imputer.fit_transform(df_model[num_cols])

print(f"\nJumlah baris setelah pembersihan missing value: {len(df_model)}")
print(f"Jumlah total missing value tersisa: {df_model.isnull().sum().sum()}")

Jumlah missing values sebelum penanganan:
income_ratio             635
bmi                      312
waist_circ               600
systolic_bp              566
diastolic_bp             566
hypertension               6
diabetes                   3
heart_failure             11
coronary_disease          25
heart_attack               8
ever_smoked               10
current_smoker          3330
sleep_hours               32
sleep_problem_doctor       4
stress_score             806
stress_anhedonia         813
stress_depressed         803
stress_fatigue           803
stress_self_esteem       805
vigorous_work              2
vigorous_work_min       4549
vigorous_leisure           1
moderate_leisure           4
sedentary_min             11
dtype: int64

Jumlah baris setelah pembersihan missing value: 5090
Jumlah total missing value tersisa: 0


### D. Pembagian Data, Penskalaan, dan SMOTE

In [9]:
# Memisahkan fitur dan target
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET].astype(int)

# Membagi data menjadi train set (80%) dan test set (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Sebelum resampling (Training Set):")
print(y_train.value_counts().to_dict())

# Menerapkan SMOTE pada training set untuk menangani data imbalance
smote = SMOTE(random_state=42, k_neighbors=3)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Setelah resampling SMOTE (Training Set):")
print(pd.Series(y_train_resampled).value_counts().to_dict())

# Penskalaan fitur menggunakan StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_resampled)
X_test_scaled = scaler.transform(X_test)

print("\nBentuk data training setelah preprocessing:", X_train_scaled.shape)
print("Bentuk data testing setelah preprocessing:", X_test_scaled.shape)
print("Pembersihan dan preprocessing dataset selesai.")

Sebelum resampling (Training Set):
{0: 3930, 1: 142}
Setelah resampling SMOTE (Training Set):
{0: 3930, 1: 3930}

Bentuk data training setelah preprocessing: (7860, 31)
Bentuk data testing setelah preprocessing: (1018, 31)
Pembersihan dan preprocessing dataset selesai.
